#  VSHORAD - Section 6.3: System Performance (Latency Benchmark)

**Goal**: Comprehensive performance benchmarks for all VSHORAD system tiers.

## Test Overview

| Tier | YOLO | imgsz | SWIN | imgsz | Backend |
|------|------|-------|------|-------|----------|
| **Strategic** | YOLOv8l | 1280 | Swin-Base | 384 | PyTorch |
| **Tactical** | YOLOv8m | 960 | Swin-Small | 224 | PyTorch |
| **Embedded** | YOLOv8m | 640 | Swin-Small | 224 | TensorRT/ONNX |

## Metrics

1. **Individual components**: FPS, latency (avg/p50/p95/p99), VRAM
2. **End-to-end pipeline**: Full system FPS on a 10-minute video
3. **Scalability**: Latency vs number of objects per frame

# 1. Setup

In [ ]:
# 1.1 Install dependencies

print(" Cleaning environment...")
!pip uninstall numpy ultralytics -y -q 2>/dev/null

print(" [1/4] Installing NumPy...")
!pip install numpy==1.26.4 -q

print(" [2/4] Installing OpenCV...")
!pip install opencv-python-headless<4.10 -q

print(" [3/4] Installing Ultralytics + TIMM...")
!pip install ultralytics==8.3.0 timm==1.0.11 albumentations==1.4.18 lapx>=0.5.2 -q

print(" [4/4] Installing ONNX Runtime GPU...")
!pip install onnxruntime-gpu -q

print("\n Installation complete!")

In [ ]:
# 1.2 Imports

import os
import sys
import json
import time
import warnings
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from typing import Dict, List, Tuple, Optional
import gc

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from ultralytics import YOLO

try:
  import onnxruntime as ort
  HAS_ONNX = True
except ImportError:
  HAS_ONNX = False

warnings.filterwarnings('ignore')

# Matplotlib config
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
sns.set_style("whitegrid")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f" Imports loaded!")
print(f"  Device: {DEVICE}")
if torch.cuda.is_available():
  print(f"  GPU: {torch.cuda.get_device_name(0)}")
  print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
  print(f"  CUDA: {torch.version.cuda}")
print(f"  ONNX Runtime: {'' if HAS_ONNX else ''}")

In [ ]:
# 1.3 Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

print(" Google Drive mounted.")

# 2. Config

In [ ]:
# 2.1 Model and data paths

DRIVE_ROOT = Path('/content/drive/MyDrive/Inżynierka')

# STRATEGIC TIER (YOLOv8l@1280 + Swin-Base@384)
STRATEGIC_ROOT = DRIVE_ROOT / 'Strategic_components'
STRATEGIC_YOLO = STRATEGIC_ROOT / 'checkpoints' / 'yolo' / 'strategic_yolov8l_1280' / 'weights' / 'best.pt'
STRATEGIC_SWIN = STRATEGIC_ROOT / 'checkpoints' / 'swin' / 'strategic_swin_base_384' / 'best.pth'

# TACTICAL TIER (YOLOv8m@960 + Swin-Small@224)
TACTICAL_ROOT = DRIVE_ROOT / 'Medium_components'
TACTICAL_YOLO = TACTICAL_ROOT / 'checkpoints' / 'yolo' / 'medium_yolov8m_960' / 'weights' / 'best.pt'
TACTICAL_SWIN = TACTICAL_ROOT / 'checkpoints' / 'swin' / 'medium_swin_small_224' / 'best.pth'

# EMBEDDED TIER (YOLOv8m@640 TensorRT + Swin-Small@224 ONNX)
EMBEDDED_ROOT = DRIVE_ROOT / 'Small_components'
EMBEDDED_EXPORTS = EMBEDDED_ROOT / 'exports'

# YOLO - priorytet: TensorRT > ONNX > PyTorch
EMBEDDED_YOLO_CANDIDATES = [
  EMBEDDED_EXPORTS / 'yolo' / 'yolov8m_640_tensorrt' / 'yolov8m_640_fp16.engine',
  EMBEDDED_EXPORTS / 'yolo' / 'small_yolov8m_640_int8' / 'yolov8m_640_fp32.onnx',
  TACTICAL_YOLO, # fallback
]

# SWIN - priorytet: ONNX merged > TensorRT > PyTorch
EMBEDDED_SWIN_CANDIDATES = [
  EMBEDDED_EXPORTS / 'swin' / 'swin_small_224_onnx' / 'swin_small_224_fp32_merged.onnx',
  EMBEDDED_EXPORTS / 'swin' / 'swin_small_224_tensorrt' / 'swin_small_224_fp16.engine',
  TACTICAL_SWIN, # fallback
]

# WIDEO TESTOWE
# Change to your test video!
TEST_VIDEO_PATH = DRIVE_ROOT / 'filmy_testowe' / 'Defilada_1.mp4'

# Alternatywne lokalizacje
VIDEO_CANDIDATES = [
  TEST_VIDEO_PATH,
  DRIVE_ROOT / 'Strategic_components' / 'test_videos' / 'demo.mp4',
  DRIVE_ROOT / 'Medium_components' / 'test_videos' / 'demo.mp4',
  DRIVE_ROOT / 'filmy' / 'test.mp4',
]

# OUTPUT
OUTPUT_DIR = DRIVE_ROOT / 'Raporty' / 'Latency'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = OUTPUT_DIR / f'benchmark_{TIMESTAMP}'
RUN_DIR.mkdir(exist_ok=True)

print(" Path config:")
print(f"  Output: {RUN_DIR}")

In [ ]:
# 2.2 Verify models and video

def find_first_existing(candidates: List[Path], name: str) -> Optional[Path]:
  """Find first existing file from candidate list."""
  for path in candidates:
    if path.exists():
      return path
  return None

print(" Verifying models...")
print()

# Strategic
print(" STRATEGIC TIER:")
strategic_ok = True
if STRATEGIC_YOLO.exists():
  print(f"  YOLO: {STRATEGIC_YOLO.name}")
else:
  print(f"  YOLO: BRAK - {STRATEGIC_YOLO}")
  strategic_ok = False

if STRATEGIC_SWIN.exists():
  print(f"  SWIN: {STRATEGIC_SWIN.name}")
else:
  print(f"  SWIN: BRAK - {STRATEGIC_SWIN}")
  strategic_ok = False

# Tactical
print("\n TACTICAL TIER:")
tactical_ok = True
if TACTICAL_YOLO.exists():
  print(f"  YOLO: {TACTICAL_YOLO.name}")
else:
  print(f"  YOLO: BRAK - {TACTICAL_YOLO}")
  tactical_ok = False

if TACTICAL_SWIN.exists():
  print(f"  SWIN: {TACTICAL_SWIN.name}")
else:
  print(f"  SWIN: BRAK - {TACTICAL_SWIN}")
  tactical_ok = False

# Embedded
print("\n EMBEDDED TIER:")
EMBEDDED_YOLO = find_first_existing(EMBEDDED_YOLO_CANDIDATES, "YOLO")
EMBEDDED_SWIN = find_first_existing(EMBEDDED_SWIN_CANDIDATES, "SWIN")

embedded_ok = True
if EMBEDDED_YOLO:
  print(f"  YOLO: {EMBEDDED_YOLO.name} ({EMBEDDED_YOLO.suffix})")
else:
  print(f"  YOLO: No candidate found!")
  embedded_ok = False

if EMBEDDED_SWIN:
  print(f"  SWIN: {EMBEDDED_SWIN.name} ({EMBEDDED_SWIN.suffix})")
else:
  print(f"  SWIN: No candidate found!")
  embedded_ok = False

# Video
print("\n WIDEO TESTOWE:")
TEST_VIDEO = find_first_existing(VIDEO_CANDIDATES, "Video")
if TEST_VIDEO:
  cap = cv2.VideoCapture(str(TEST_VIDEO))
  fps = cap.get(cv2.CAP_PROP_FPS)
  frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
  width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
  height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
  duration = frames / fps if fps > 0 else 0
  cap.release()
  print(f"  {TEST_VIDEO.name}")
  print(f"   Resolution: {width}x{height}")
  print(f"   FPS: {fps:.1f}")
  print(f"   Klatki: {frames}")
  print(f"   Czas: {duration/60:.1f} min")
else:
  print(f"  Nie znaleziono wideo testowego!")
  print(f"   Check one of these locations:")
  for v in VIDEO_CANDIDATES:
    print(f"   - {v}")

# Summary
print("\n" + "="*60)
all_ok = strategic_ok and tactical_ok and embedded_ok and TEST_VIDEO
if all_ok:
  print(" ALL MODELS AND VIDEO READY!")
else:
  print(" MISSING FILES - check paths above")

In [ ]:
# 2.3 Benchmark config

# Test parameters
BENCHMARK_CONFIG = {
  # Component benchmarks
  'component_warmup_iterations': 50,   # Rozgrzewka przed pomiarem
  'component_test_iterations': 500,    # Ile iteracji do pomiaru
  'component_batch_size': 1,       # Batch size (1 = real-time)

  # Benchmark pipeline na wideo
  'video_max_frames': None,        # None = full video, or frame count
  'video_skip_frames': 0,         # Skip every N frames (0 = all)

  # YOLO
  'yolo_conf_threshold': 0.25,
  'yolo_iou_threshold': 0.45,

  # Re-ID params (same for all tiers)
  'reid_max_age': 30,
  'reid_min_hits': 3,
  'reid_iou_threshold': 0.3,
}

# Tier configs
TIER_CONFIGS = {
  'strategic': {
    'name': 'Strategic',
    'yolo_path': STRATEGIC_YOLO,
    'yolo_imgsz': 1280,
    'swin_path': STRATEGIC_SWIN,
    'swin_imgsz': 384,
    'swin_model_name': 'swin_base_patch4_window12_384',
    'backend': 'pytorch',
  },
  'tactical': {
    'name': 'Tactical',
    'yolo_path': TACTICAL_YOLO,
    'yolo_imgsz': 960,
    'swin_path': TACTICAL_SWIN,
    'swin_imgsz': 224,
    'swin_model_name': 'swin_small_patch4_window7_224',
    'backend': 'pytorch',
  },
  'embedded': {
    'name': 'Embedded',
    'yolo_path': EMBEDDED_YOLO,
    'yolo_imgsz': 640,
    'swin_path': EMBEDDED_SWIN,
    'swin_imgsz': 224,
    'swin_model_name': 'swin_small_patch4_window7_224',
    'backend': 'tensorrt/onnx',
  },
}

# Mapowanie classes YOLO -> SWIN
SWIN_CLASSES = [
  'A-10', 'A400M', 'AG600', 'AV-8B', 'An-124', 'An-22', 'An-225', 'An-72',
  'B-1', 'B-2', 'B-52', 'Be-200', 'C-130', 'C-17', 'C-2', 'C-5', 'E-2',
  'E-7', 'EF2000', 'F-117', 'F-14', 'F-15', 'F-16', 'F-18', 'F-22', 'F-35',
  'F-4', 'H-6', 'J-20', 'JAS39', 'JF-17', 'KC-135', 'MQ-9', 'Mi-24', 'Mi-26',
  'Mi-28', 'Mig-29', 'Mig-31', 'Mirage2000', 'P-3', 'Rafale', 'RQ-4',
  'SR-71', 'Su-24', 'Su-25', 'Su-27', 'Su-34', 'Su-57', 'Tornado', 'Tu-160',
  'Tu-22M', 'Tu-95', 'U-2', 'UH-60', 'V-22', 'XB-70'
]

YOLO_TO_SWIN_MAP = {
  'Fighter': [i for i, c in enumerate(SWIN_CLASSES) if c in [
    'F-14', 'F-15', 'F-16', 'F-18', 'F-22', 'F-35', 'F-4', 'F-117',
    'EF2000', 'JAS39', 'JF-17', 'Mig-29', 'Mig-31', 'Mirage2000',
    'Rafale', 'Su-27', 'Su-57', 'J-20', 'Tornado'
  ]],
  'Bomber': [i for i, c in enumerate(SWIN_CLASSES) if c in [
    'B-1', 'B-2', 'B-52', 'Tu-160', 'Tu-22M', 'Tu-95', 'H-6', 'XB-70'
  ]],
  'Transport': [i for i, c in enumerate(SWIN_CLASSES) if c in [
    'C-130', 'C-17', 'C-2', 'C-5', 'An-124', 'An-22', 'An-225',
    'An-72', 'A400M', 'KC-135'
  ]],
  'Helicopter': [i for i, c in enumerate(SWIN_CLASSES) if c in [
    'Mi-24', 'Mi-26', 'Mi-28', 'UH-60', 'V-22'
  ]],
  'Special': [i for i, c in enumerate(SWIN_CLASSES) if c in [
    'E-2', 'E-7', 'P-3', 'U-2', 'SR-71', 'Be-200', 'AG600', 'A-10',
    'AV-8B', 'Su-24', 'Su-25', 'Su-34'
  ]],
  'UAV_Fixed': [i for i, c in enumerate(SWIN_CLASSES) if c in [
    'MQ-9', 'RQ-4'
  ]],
}

print(" Benchmark config loaded")
print(f"  Warmup: {BENCHMARK_CONFIG['component_warmup_iterations']} iteracji")
print(f"  Test: {BENCHMARK_CONFIG['component_test_iterations']} iteracji")
print(f"  SWIN classes: {len(SWIN_CLASSES)}")

# 3. Helper classes and utilities

In [ ]:
# 3.1 LATENCY TRACKER

class LatencyTracker:
  """Tracker for collecting and analyzing latency."""

  def __init__(self, name: str):
    self.name = name
    self.latencies = []
    self.frame_data = [] # (frame_idx, num_objects, latency)

  def add(self, latency_ms: float, frame_idx: int = -1, num_objects: int = 0):
    """Dodaj pomiar latencji."""
    self.latencies.append(latency_ms)
    if frame_idx >= 0:
      self.frame_data.append((frame_idx, num_objects, latency_ms))

  def reset(self):
    """Resetuj pomiary."""
    self.latencies = []
    self.frame_data = []

  def get_stats(self) -> Dict:
    """Return statistics."""
    if not self.latencies:
      return {}

    arr = np.array(self.latencies)
    return {
      'name': self.name,
      'count': len(arr),
      'mean_ms': float(np.mean(arr)),
      'std_ms': float(np.std(arr)),
      'min_ms': float(np.min(arr)),
      'max_ms': float(np.max(arr)),
      'p50_ms': float(np.percentile(arr, 50)),
      'p95_ms': float(np.percentile(arr, 95)),
      'p99_ms': float(np.percentile(arr, 99)),
      'fps_mean': 1000.0 / np.mean(arr) if np.mean(arr) > 0 else 0,
      'fps_p50': 1000.0 / np.percentile(arr, 50) if np.percentile(arr, 50) > 0 else 0,
    }

  def get_frame_df(self) -> pd.DataFrame:
    """Return per-frame DataFrame."""
    if not self.frame_data:
      return pd.DataFrame()
    return pd.DataFrame(self.frame_data, columns=['frame_idx', 'num_objects', 'latency_ms'])

def get_gpu_memory_mb() -> float:
  """Return current GPU memory usage in MB."""
  if torch.cuda.is_available():
    return torch.cuda.memory_allocated() / 1024 / 1024
  return 0.0

def get_gpu_memory_peak_mb() -> float:
  """Return peak GPU memory usage in MB."""
  if torch.cuda.is_available():
    return torch.cuda.max_memory_allocated() / 1024 / 1024
  return 0.0

def clear_gpu_memory():
  """Clear GPU memory."""
  if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
  gc.collect()

print(" LatencyTracker zdefiniowany")

In [ ]:
# 3.2 SWIN TRANSFORMER WRAPPER

class SwinClassifier:
  """Wrapper dla Swin Transformer (PyTorch)."""

  def __init__(self, model_path: Path, model_name: str, img_size: int,
         num_classes: int = 56, device: str = 'cuda'):
    self.device = device
    self.img_size = img_size
    self.num_classes = num_classes

    print(f"  Loading Swin: {model_name}...")

    # Create model
    self.model = timm.create_model(
      model_name,
      pretrained=False,
      num_classes=num_classes,
      img_size=img_size
    )

    # Load weights
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    if 'model_state_dict' in checkpoint:
      self.model.load_state_dict(checkpoint['model_state_dict'])
    else:
      self.model.load_state_dict(checkpoint)

    self.model = self.model.to(device)
    self.model.eval()

    # Transformacje
    self.transform = A.Compose([
      A.Resize(img_size, img_size),
      A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
      ToTensorV2()
    ])

    print(f"  Swin loaded! (img_size={img_size})")

  def preprocess(self, image: np.ndarray) -> torch.Tensor:
    """Prepare image for classification."""
    if image.shape[2] == 4: # BGRA
      image = cv2.cvtColor(image, cv2.COLOR_BGRA2RGB)
    elif len(image.shape) == 2: # Grayscale
      image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    else:
      image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    transformed = self.transform(image=image)
    return transformed['image'].unsqueeze(0).to(self.device)

  @torch.no_grad()
  def predict(self, image: np.ndarray) -> Tuple[int, float, np.ndarray]:
    """Classify image. Returns (class_idx, confidence, all_probs)."""
    tensor = self.preprocess(image)
    logits = self.model(tensor)
    probs = F.softmax(logits, dim=1)
    conf, pred = probs.max(dim=1)
    return pred.item(), conf.item(), probs.cpu().numpy()[0]

  @torch.no_grad()
  def predict_batch(self, images: List[np.ndarray]) -> List[Tuple[int, float]]:
    """Classify batch of images."""
    if not images:
      return []

    tensors = torch.cat([self.preprocess(img) for img in images], dim=0)
    logits = self.model(tensors)
    probs = F.softmax(logits, dim=1)
    confs, preds = probs.max(dim=1)

    return list(zip(preds.cpu().tolist(), confs.cpu().tolist()))

class SwinONNXClassifier:
  """Wrapper dla Swin Transformer (ONNX Runtime)."""

  def __init__(self, model_path: Path, img_size: int = 224):
    self.img_size = img_size

    print(f"  Loading Swin ONNX: {model_path.name}...")

    # ONNX Runtime config
    providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
    sess_options = ort.SessionOptions()
    sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

    self.session = ort.InferenceSession(
      str(model_path),
      sess_options=sess_options,
      providers=providers
    )

    self.input_name = self.session.get_inputs()[0].name
    self.output_name = self.session.get_outputs()[0].name

    # Normalizacja
    self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    self.std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    print(f"  Swin ONNX loaded!")

  def preprocess(self, image: np.ndarray) -> np.ndarray:
    """Prepare image for classification."""
    if len(image.shape) == 2:
      image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    else:
      image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    image = cv2.resize(image, (self.img_size, self.img_size))
    image = image.astype(np.float32) / 255.0
    image = (image - self.mean) / self.std
    image = image.transpose(2, 0, 1) # HWC -> CHW
    return image[np.newaxis, ...] # Add batch dim

  def predict(self, image: np.ndarray) -> Tuple[int, float, np.ndarray]:
    """Classify image."""
    input_tensor = self.preprocess(image)
    outputs = self.session.run([self.output_name], {self.input_name: input_tensor})
    logits = outputs[0][0]

    # Softmax
    exp_logits = np.exp(logits - np.max(logits))
    probs = exp_logits / exp_logits.sum()

    pred_idx = np.argmax(probs)
    conf = probs[pred_idx]

    return int(pred_idx), float(conf), probs

print(" Classes Swin zdefiniowane")

# 4. Component benchmarks

In [ ]:
# 4.1 YOLO benchmark function

def benchmark_yolo(model_path: Path, imgsz: int, tier_name: str,
          warmup: int = 50, iterations: int = 500) -> Dict:
  """Benchmark YOLO model."""

  print(f"\n Benchmark YOLO - {tier_name}")
  print(f"  Model: {model_path.name}")
  print(f"  Image size: {imgsz}")

  clear_gpu_memory()

  # Load model
  print(f"  Loading model...")
  model = YOLO(str(model_path))

  # Generuj syntetyczny obraz
  test_image = np.random.randint(0, 255, (imgsz, imgsz, 3), dtype=np.uint8)

  # Warmup
  print(f"  Warmup ({warmup} iteracji)...")
  for _ in tqdm(range(warmup), desc="Warmup", leave=False):
    _ = model.predict(
      test_image,
      imgsz=imgsz,
      conf=BENCHMARK_CONFIG['yolo_conf_threshold'],
      iou=BENCHMARK_CONFIG['yolo_iou_threshold'],
      verbose=False
    )

  if torch.cuda.is_available():
    torch.cuda.synchronize()

  # Benchmark
  print(f"   Benchmark ({iterations} iteracji)...")
  tracker = LatencyTracker(f"YOLO_{tier_name}")

  for i in tqdm(range(iterations), desc="Benchmark", leave=False):
    if torch.cuda.is_available():
      torch.cuda.synchronize()

    start = time.perf_counter()
    results = model.predict(
      test_image,
      imgsz=imgsz,
      conf=BENCHMARK_CONFIG['yolo_conf_threshold'],
      iou=BENCHMARK_CONFIG['yolo_iou_threshold'],
      verbose=False
    )

    if torch.cuda.is_available():
      torch.cuda.synchronize()

    end = time.perf_counter()
    tracker.add((end - start) * 1000)

  # Zbierz statystyki
  stats = tracker.get_stats()
  stats['tier'] = tier_name
  stats['model'] = model_path.name
  stats['imgsz'] = imgsz
  stats['vram_peak_mb'] = get_gpu_memory_peak_mb()

  # Print results
  print(f"\n  YOLO results {tier_name}:")
  print(f"   Mean: {stats['mean_ms']:.2f} ms ({stats['fps_mean']:.1f} FPS)")
  print(f"   P50: {stats['p50_ms']:.2f} ms ({stats['fps_p50']:.1f} FPS)")
  print(f"   P95: {stats['p95_ms']:.2f} ms")
  print(f"   P99: {stats['p99_ms']:.2f} ms")
  print(f"   VRAM: {stats['vram_peak_mb']:.0f} MB")

  # Cleanup
  del model
  clear_gpu_memory()

  return stats

print(" Funkcja benchmark_yolo zdefiniowana")

In [ ]:
# 4.2 Swin benchmark function

def benchmark_swin(model_path: Path, model_name: str, imgsz: int,
          tier_name: str, backend: str = 'pytorch',
          warmup: int = 50, iterations: int = 500) -> Dict:
  """Benchmark Swin Transformer model."""

  print(f"\n Benchmark SWIN - {tier_name}")
  print(f"  Model: {model_path.name}")
  print(f"  Image size: {imgsz}")
  print(f"  Backend: {backend}")

  clear_gpu_memory()

  # Load model
  print(f"  Loading model...")

  if model_path.suffix == '.onnx' and HAS_ONNX:
    classifier = SwinONNXClassifier(model_path, img_size=imgsz)
    actual_backend = 'onnx'
  elif model_path.suffix == '.engine':
    # TensorRT - use ONNX as fallback
    print(f"  TensorRT engine requires special handling, looking for ONNX...")
    onnx_path = model_path.parent.parent / 'swin_small_224_onnx' / 'swin_small_224_fp32_merged.onnx'
    if onnx_path.exists() and HAS_ONNX:
      classifier = SwinONNXClassifier(onnx_path, img_size=imgsz)
      actual_backend = 'onnx'
    else:
      classifier = SwinClassifier(model_path, model_name, imgsz)
      actual_backend = 'pytorch'
  else:
    classifier = SwinClassifier(model_path, model_name, imgsz)
    actual_backend = 'pytorch'

  # Generuj syntetyczny obraz (typowy crop z YOLO)
  test_image = np.random.randint(0, 255, (imgsz, imgsz, 3), dtype=np.uint8)

  # Warmup
  print(f"  Warmup ({warmup} iteracji)...")
  for _ in tqdm(range(warmup), desc="Warmup", leave=False):
    _ = classifier.predict(test_image)

  if torch.cuda.is_available():
    torch.cuda.synchronize()

  # Benchmark
  print(f"   Benchmark ({iterations} iteracji)...")
  tracker = LatencyTracker(f"SWIN_{tier_name}")

  for i in tqdm(range(iterations), desc="Benchmark", leave=False):
    if torch.cuda.is_available():
      torch.cuda.synchronize()

    start = time.perf_counter()
    _ = classifier.predict(test_image)

    if torch.cuda.is_available():
      torch.cuda.synchronize()

    end = time.perf_counter()
    tracker.add((end - start) * 1000)

  # Zbierz statystyki
  stats = tracker.get_stats()
  stats['tier'] = tier_name
  stats['model'] = model_path.name
  stats['imgsz'] = imgsz
  stats['backend'] = actual_backend
  stats['vram_peak_mb'] = get_gpu_memory_peak_mb()

  # Print results
  print(f"\n  SWIN results {tier_name}:")
  print(f"   Mean: {stats['mean_ms']:.2f} ms ({stats['fps_mean']:.1f} FPS)")
  print(f"   P50: {stats['p50_ms']:.2f} ms ({stats['fps_p50']:.1f} FPS)")
  print(f"   P95: {stats['p95_ms']:.2f} ms")
  print(f"   P99: {stats['p99_ms']:.2f} ms")
  print(f"   VRAM: {stats['vram_peak_mb']:.0f} MB")

  # Cleanup
  del classifier
  clear_gpu_memory()

  return stats

print(" Funkcja benchmark_swin zdefiniowana")

In [ ]:
# 4.3 Run component benchmarks

print("="*70)
print(" Component benchmarks")
print("="*70)

component_results = {
  'yolo': [],
  'swin': [],
}

# YOLO BENCHMARKS
print("\n" + ""*70)
print(" YOLO MODELS")
print(""*70)

for tier_key, config in TIER_CONFIGS.items():
  if config['yolo_path'] and config['yolo_path'].exists():
    stats = benchmark_yolo(
      model_path=config['yolo_path'],
      imgsz=config['yolo_imgsz'],
      tier_name=config['name'],
      warmup=BENCHMARK_CONFIG['component_warmup_iterations'],
      iterations=BENCHMARK_CONFIG['component_test_iterations']
    )
    component_results['yolo'].append(stats)
  else:
    print(f"\n Skipping {config['name']} YOLO - model not found")

# SWIN BENCHMARKS
print("\n" + ""*70)
print(" SWIN MODELS")
print(""*70)

for tier_key, config in TIER_CONFIGS.items():
  if config['swin_path'] and config['swin_path'].exists():
    stats = benchmark_swin(
      model_path=config['swin_path'],
      model_name=config['swin_model_name'],
      imgsz=config['swin_imgsz'],
      tier_name=config['name'],
      backend=config['backend'],
      warmup=BENCHMARK_CONFIG['component_warmup_iterations'],
      iterations=BENCHMARK_CONFIG['component_test_iterations']
    )
    component_results['swin'].append(stats)
  else:
    print(f"\n Skipping {config['name']} SWIN - model not found")

print("\n" + "="*70)
print(" COMPONENT BENCHMARK COMPLETE")
print("="*70)

# 5. End-to-end pipeline benchmark

In [ ]:
# 5.1 VSHORAD system class (simplified for benchmarking)

class VSHORADBenchmarkSystem:
  """
  Simplified VSHORAD system for latency benchmarking.
  Contains: YOLO -> ByteTrack -> Swin (without full RE-ID for simplicity).
  """

  def __init__(self, tier_config: Dict):
    self.tier_name = tier_config['name']
    self.config = tier_config

    print(f"\n Inicjalizacja systemu {self.tier_name}...")

    # Load YOLO
    print(f"  [1/2] Loading YOLO...")
    self.yolo = YOLO(str(tier_config['yolo_path']))
    self.yolo_imgsz = tier_config['yolo_imgsz']

    # Load SWIN
    print(f"  [2/2] Loading SWIN...")
    swin_path = tier_config['swin_path']

    if swin_path.suffix == '.onnx' and HAS_ONNX:
      self.swin = SwinONNXClassifier(swin_path, img_size=tier_config['swin_imgsz'])
      self.swin_backend = 'onnx'
    else:
      self.swin = SwinClassifier(
        swin_path,
        tier_config['swin_model_name'],
        tier_config['swin_imgsz']
      )
      self.swin_backend = 'pytorch'

    # Latency trackers
    self.tracker_total = LatencyTracker(f"Pipeline_{self.tier_name}")
    self.tracker_yolo = LatencyTracker(f"YOLO_{self.tier_name}")
    self.tracker_swin = LatencyTracker(f"SWIN_{self.tier_name}")

    print(f"  System {self.tier_name} gotowy!")

  def process_frame(self, frame: np.ndarray, frame_idx: int = 0) -> Dict:
    """
    Process a single frame.
    Returns dict with results and latencies.
    """
    total_start = time.perf_counter()

    # YOLO Detection
    if torch.cuda.is_available():
      torch.cuda.synchronize()

    yolo_start = time.perf_counter()
    results = self.yolo.predict(
      frame,
      imgsz=self.yolo_imgsz,
      conf=BENCHMARK_CONFIG['yolo_conf_threshold'],
      iou=BENCHMARK_CONFIG['yolo_iou_threshold'],
      verbose=False
    )

    if torch.cuda.is_available():
      torch.cuda.synchronize()
    yolo_end = time.perf_counter()

    yolo_latency = (yolo_end - yolo_start) * 1000

    # Parse results
    detections = []
    if len(results) > 0 and results[0].boxes is not None:
      boxes = results[0].boxes
      for i in range(len(boxes)):
        x1, y1, x2, y2 = boxes.xyxy[i].cpu().numpy()
        conf = float(boxes.conf[i].cpu().numpy())
        cls = int(boxes.cls[i].cpu().numpy())
        detections.append({
          'bbox': [int(x1), int(y1), int(x2), int(y2)],
          'confidence': conf,
          'class_id': cls
        })

    num_objects = len(detections)

    # SWIN Classification (for each detection)
    if torch.cuda.is_available():
      torch.cuda.synchronize()

    swin_start = time.perf_counter()

    classifications = []
    for det in detections:
      x1, y1, x2, y2 = det['bbox']
      # Wytnij crop
      crop = frame[max(0, y1):y2, max(0, x1):x2]
      if crop.size > 0:
        pred_idx, conf, _ = self.swin.predict(crop)
        classifications.append({
          'class_idx': pred_idx,
          'confidence': conf
        })

    if torch.cuda.is_available():
      torch.cuda.synchronize()
    swin_end = time.perf_counter()

    swin_latency = (swin_end - swin_start) * 1000

    # Total
    total_end = time.perf_counter()
    total_latency = (total_end - total_start) * 1000

    # Save latencies
    self.tracker_total.add(total_latency, frame_idx, num_objects)
    self.tracker_yolo.add(yolo_latency, frame_idx, num_objects)
    self.tracker_swin.add(swin_latency, frame_idx, num_objects)

    return {
      'frame_idx': frame_idx,
      'num_objects': num_objects,
      'latency_total_ms': total_latency,
      'latency_yolo_ms': yolo_latency,
      'latency_swin_ms': swin_latency,
      'detections': len(detections),
      'classifications': len(classifications)
    }

  def get_results(self) -> Dict:
    """Return all benchmark results."""
    return {
      'tier': self.tier_name,
      'total': self.tracker_total.get_stats(),
      'yolo': self.tracker_yolo.get_stats(),
      'swin': self.tracker_swin.get_stats(),
      'frame_data': self.tracker_total.get_frame_df().to_dict('records')
    }

  def cleanup(self):
    """Release resources."""
    del self.yolo
    del self.swin
    clear_gpu_memory()

print(" VSHORADBenchmarkSystem zdefiniowany")

In [ ]:
# 5.2 Run pipeline benchmark on video

print("="*70)
print(" End-to-end pipeline benchmark")
print("="*70)

if TEST_VIDEO is None or not TEST_VIDEO.exists():
  print("\n Brak wideo testowego! Skipping benchmark pipeline.")
  pipeline_results = []
else:
  pipeline_results = []

  # Open video
  cap = cv2.VideoCapture(str(TEST_VIDEO))
  total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
  fps_video = cap.get(cv2.CAP_PROP_FPS)

  max_frames = BENCHMARK_CONFIG.get('video_max_frames') or total_frames
  skip_frames = BENCHMARK_CONFIG.get('video_skip_frames', 0)

  print(f"\n Wideo: {TEST_VIDEO.name}")
  print(f"  Klatki: {total_frames} (testujemy: {max_frames})")
  print(f"  FPS wideo: {fps_video:.1f}")

  # Load all frames into memory (fair comparison)
  print(f"\n Loading frames into memory...")
  frames = []
  for i in tqdm(range(max_frames), desc="Loading frames"):
    ret, frame = cap.read()
    if not ret:
      break
    if skip_frames > 0 and i % (skip_frames + 1) != 0:
      continue
    frames.append(frame)
  cap.release()

  print(f"  Loaded {len(frames)} frames")

  # Benchmark each tier
  for tier_key, config in TIER_CONFIGS.items():
    if not config['yolo_path'] or not config['yolo_path'].exists():
      print(f"\n Skipping {config['name']} - model not found YOLO")
      continue
    if not config['swin_path'] or not config['swin_path'].exists():
      print(f"\n Skipping {config['name']} - model not found SWIN")
      continue

    print(f"\n" + ""*70)
    print(f" BENCHMARK PIPELINE: {config['name']}")
    print(""*70)

    # Inicjalizuj system
    system = VSHORADBenchmarkSystem(config)

    # Warmup (10 frames)
    print(f"\n  Warmup (10 frames)...")
    for i in range(min(10, len(frames))):
      _ = system.process_frame(frames[i], frame_idx=i)
    system.tracker_total.reset()
    system.tracker_yolo.reset()
    system.tracker_swin.reset()

    # Benchmark
    print(f"   Processing {len(frames)} frames...")
    for i in tqdm(range(len(frames)), desc=f"{config['name']}"):
      _ = system.process_frame(frames[i], frame_idx=i)

    # Collect results
    results = system.get_results()
    results['config'] = {
      'yolo_imgsz': config['yolo_imgsz'],
      'swin_imgsz': config['swin_imgsz'],
      'backend': config['backend'],
    }
    pipeline_results.append(results)

    # Print results
    total_stats = results['total']
    print(f"\n  Results {config['name']}:")
    print(f"   Total Mean: {total_stats['mean_ms']:.2f} ms ({total_stats['fps_mean']:.1f} FPS)")
    print(f"   Total P50: {total_stats['p50_ms']:.2f} ms ({total_stats['fps_p50']:.1f} FPS)")
    print(f"   Total P95: {total_stats['p95_ms']:.2f} ms")
    print(f"   YOLO Mean: {results['yolo']['mean_ms']:.2f} ms")
    print(f"   SWIN Mean: {results['swin']['mean_ms']:.2f} ms")

    # Cleanup
    system.cleanup()

  # Free frame memory
  del frames
  gc.collect()

print("\n" + "="*70)
print(" PIPELINE BENCHMARK COMPLETE")
print("="*70)

# 6. Generate Plots & Report

In [ ]:
# 6.1 Generate plots

print(" Generating plots...")

CHARTS_DIR = RUN_DIR / 'plots'
CHARTS_DIR.mkdir(exist_ok=True)

# Colors per tier
TIER_COLORS = {
  'Strategic': '#2ecc71', # zielony
  'Tactical': '#3498db',  # niebieski
  'Embedded': '#e74c3c',  # czerwony
}

# Chart 1: Component FPS comparison
if component_results['yolo'] or component_results['swin']:
  fig, axes = plt.subplots(1, 2, figsize=(14, 6))

  # YOLO FPS
  if component_results['yolo']:
    yolo_data = component_results['yolo']
    tiers = [d['tier'] for d in yolo_data]
    fps_mean = [d['fps_mean'] for d in yolo_data]
    colors = [TIER_COLORS.get(t, '#95a5a6') for t in tiers]

    bars = axes[0].bar(tiers, fps_mean, color=colors, edgecolor='black', linewidth=1.2)
    axes[0].set_title('YOLO - FPS (higher = better)', fontsize=13, fontweight='bold')
    axes[0].set_ylabel('FPS')
    axes[0].set_ylim(0, max(fps_mean) * 1.2)

    for bar, fps in zip(bars, fps_mean):
      axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f'{fps:.1f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

  # SWIN FPS
  if component_results['swin']:
    swin_data = component_results['swin']
    tiers = [d['tier'] for d in swin_data]
    fps_mean = [d['fps_mean'] for d in swin_data]
    colors = [TIER_COLORS.get(t, '#95a5a6') for t in tiers]

    bars = axes[1].bar(tiers, fps_mean, color=colors, edgecolor='black', linewidth=1.2)
    axes[1].set_title('SWIN - FPS (higher = better)', fontsize=13, fontweight='bold')
    axes[1].set_ylabel('FPS')
    axes[1].set_ylim(0, max(fps_mean) * 1.2)

    for bar, fps in zip(bars, fps_mean):
      axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{fps:.1f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

  plt.tight_layout()
  plt.savefig(CHARTS_DIR / 'component_fps_comparison.png', dpi=150, bbox_inches='tight')
  plt.savefig(CHARTS_DIR / 'component_fps_comparison.pdf', bbox_inches='tight')
  plt.show()
  print("  component_fps_comparison.png")

# Chart 2: Component latency percentiles (P50, P95, P99)
if component_results['yolo'] or component_results['swin']:
  fig, axes = plt.subplots(1, 2, figsize=(14, 6))

  x = np.arange(3) # P50, P95, P99
  width = 0.25

  # YOLO Latency
  if component_results['yolo']:
    for i, d in enumerate(component_results['yolo']):
      latencies = [d['p50_ms'], d['p95_ms'], d['p99_ms']]
      axes[0].bar(x + i*width, latencies, width,
            label=d['tier'], color=TIER_COLORS.get(d['tier'], '#95a5a6'),
            edgecolor='black', linewidth=1)

    axes[0].set_title('YOLO - Latency (lower = better)', fontsize=13, fontweight='bold')
    axes[0].set_ylabel('Latencja [ms]')
    axes[0].set_xticks(x + width)
    axes[0].set_xticklabels(['P50', 'P95', 'P99'])
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.3)

  # SWIN Latency
  if component_results['swin']:
    for i, d in enumerate(component_results['swin']):
      latencies = [d['p50_ms'], d['p95_ms'], d['p99_ms']]
      axes[1].bar(x + i*width, latencies, width,
            label=d['tier'], color=TIER_COLORS.get(d['tier'], '#95a5a6'),
            edgecolor='black', linewidth=1)

    axes[1].set_title('SWIN - Latency (lower = better)', fontsize=13, fontweight='bold')
    axes[1].set_ylabel('Latencja [ms]')
    axes[1].set_xticks(x + width)
    axes[1].set_xticklabels(['P50', 'P95', 'P99'])
    axes[1].legend()
    axes[1].grid(axis='y', alpha=0.3)

  plt.tight_layout()
  plt.savefig(CHARTS_DIR / 'component_latency_percentiles.png', dpi=150, bbox_inches='tight')
  plt.savefig(CHARTS_DIR / 'component_latency_percentiles.pdf', bbox_inches='tight')
  plt.show()
  print("  component_latency_percentiles.png")

# Chart 3: Pipeline FPS comparison
if pipeline_results:
  fig, ax = plt.subplots(figsize=(10, 6))

  tiers = [r['tier'] for r in pipeline_results]
  fps_mean = [r['total']['fps_mean'] for r in pipeline_results]
  fps_p50 = [r['total']['fps_p50'] for r in pipeline_results]
  colors = [TIER_COLORS.get(t, '#95a5a6') for t in tiers]

  x = np.arange(len(tiers))
  width = 0.35

  bars1 = ax.bar(x - width/2, fps_mean, width, label='Mean FPS', color=colors,
          edgecolor='black', linewidth=1.2)
  bars2 = ax.bar(x + width/2, fps_p50, width, label='P50 FPS', color=colors,
          edgecolor='black', linewidth=1.2, alpha=0.7)

  ax.set_title('Pipeline End-to-End - FPS', fontsize=14, fontweight='bold')
  ax.set_ylabel('FPS')
  ax.set_xticks(x)
  ax.set_xticklabels(tiers)
  ax.legend()
  ax.grid(axis='y', alpha=0.3)

  # Add values
  for bar, fps in zip(bars1, fps_mean):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
        f'{fps:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

  plt.tight_layout()
  plt.savefig(CHARTS_DIR / 'pipeline_fps_comparison.png', dpi=150, bbox_inches='tight')
  plt.savefig(CHARTS_DIR / 'pipeline_fps_comparison.pdf', bbox_inches='tight')
  plt.show()
  print("  pipeline_fps_comparison.png")

# Chart 4: Latency vs object count (scalability)
if pipeline_results:
  fig, ax = plt.subplots(figsize=(12, 7))

  for result in pipeline_results:
    tier = result['tier']
    frame_data = result.get('frame_data', [])

    if frame_data:
      df = pd.DataFrame(frame_data)

      # Group by object count and compute mean latency
      grouped = df.groupby('num_objects')['latency_ms'].agg(['mean', 'std', 'count'])
      grouped = grouped[grouped['count'] >= 3] # Min 3 sample

      if len(grouped) > 0:
        ax.plot(grouped.index, grouped['mean'],
            marker='o', linewidth=2, markersize=6,
            label=tier, color=TIER_COLORS.get(tier, '#95a5a6'))
        ax.fill_between(grouped.index,
                grouped['mean'] - grouped['std'],
                grouped['mean'] + grouped['std'],
                alpha=0.2, color=TIER_COLORS.get(tier, '#95a5a6'))

  ax.set_title('Latency vs Number of Objects per Frame', fontsize=14, fontweight='bold')
  ax.set_xlabel('Number of detected objects')
  ax.set_ylabel('Latencja [ms]')
  ax.legend()
  ax.grid(alpha=0.3)

  plt.tight_layout()
  plt.savefig(CHARTS_DIR / 'latency_vs_objects.png', dpi=150, bbox_inches='tight')
  plt.savefig(CHARTS_DIR / 'latency_vs_objects.pdf', bbox_inches='tight')
  plt.show()
  print("  latency_vs_objects.png")

# Chart 5: Pipeline latency distribution
if pipeline_results:
  fig, axes = plt.subplots(1, len(pipeline_results), figsize=(5*len(pipeline_results), 5))
  if len(pipeline_results) == 1:
    axes = [axes]

  for ax, result in zip(axes, pipeline_results):
    tier = result['tier']
    frame_data = result.get('frame_data', [])

    if frame_data:
      latencies = [d['latency_ms'] for d in frame_data]
      ax.hist(latencies, bins=50, color=TIER_COLORS.get(tier, '#95a5a6'),
          edgecolor='black', alpha=0.7)
      ax.axvline(np.mean(latencies), color='red', linestyle='--',
           label=f'Mean: {np.mean(latencies):.1f}ms')
      ax.axvline(np.percentile(latencies, 95), color='orange', linestyle='--',
           label=f'P95: {np.percentile(latencies, 95):.1f}ms')

      ax.set_title(f'{tier} - Latency Distribution', fontsize=12, fontweight='bold')
      ax.set_xlabel('Latencja [ms]')
      ax.set_ylabel('Frame count')
      ax.legend(fontsize=9)

  plt.tight_layout()
  plt.savefig(CHARTS_DIR / 'latency_distribution.png', dpi=150, bbox_inches='tight')
  plt.savefig(CHARTS_DIR / 'latency_distribution.pdf', bbox_inches='tight')
  plt.show()
  print("  latency_distribution.png")

# Chart 6: Pipeline latency breakdown (YOLO vs SWIN)
if pipeline_results:
  fig, ax = plt.subplots(figsize=(10, 6))

  tiers = [r['tier'] for r in pipeline_results]
  yolo_latency = [r['yolo']['mean_ms'] for r in pipeline_results]
  swin_latency = [r['swin']['mean_ms'] for r in pipeline_results]

  x = np.arange(len(tiers))
  width = 0.5

  bars1 = ax.bar(x, yolo_latency, width, label='YOLO', color='#3498db')
  bars2 = ax.bar(x, swin_latency, width, bottom=yolo_latency, label='SWIN', color='#e74c3c')

  ax.set_title('Breakdown Latencji Pipeline', fontsize=14, fontweight='bold')
  ax.set_ylabel('Latencja [ms]')
  ax.set_xticks(x)
  ax.set_xticklabels(tiers)
  ax.legend()
  ax.grid(axis='y', alpha=0.3)

  # Add values
  for i, (y, s) in enumerate(zip(yolo_latency, swin_latency)):
    ax.text(i, y/2, f'{y:.1f}', ha='center', va='center', fontsize=10, color='white', fontweight='bold')
    ax.text(i, y + s/2, f'{s:.1f}', ha='center', va='center', fontsize=10, color='white', fontweight='bold')
    ax.text(i, y + s + 1, f'Σ {y+s:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

  plt.tight_layout()
  plt.savefig(CHARTS_DIR / 'pipeline_latency_breakdown.png', dpi=150, bbox_inches='tight')
  plt.savefig(CHARTS_DIR / 'pipeline_latency_breakdown.pdf', bbox_inches='tight')
  plt.show()
  print("  pipeline_latency_breakdown.png")

print(f"\n All plots saved to: {CHARTS_DIR}")

In [ ]:
# 6.2 Generate markdown report

print(" Generating report...")

# GPU info
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
gpu_vram = f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "N/A"

report = f"""# VSHORAD - System Performance Report (Section 6.3)

**Data**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
**GPU**: {gpu_name}
**VRAM**: {gpu_vram}
**CUDA**: {torch.version.cuda if torch.cuda.is_available() else 'N/A'}

---

## 1. Test Config

| Parameter | Value |
|----------|----------|
| Warmup iterations | {BENCHMARK_CONFIG['component_warmup_iterations']} |
| Test iterations | {BENCHMARK_CONFIG['component_test_iterations']} |
| YOLO confidence | {BENCHMARK_CONFIG['yolo_conf_threshold']} |
| YOLO IoU | {BENCHMARK_CONFIG['yolo_iou_threshold']} |

### Tier Config

| Tier | YOLO | imgsz | SWIN | imgsz | Backend |
|------|------|-------|------|-------|----------|
"""

for tier_key, config in TIER_CONFIGS.items():
  yolo_name = config['yolo_path'].name if config['yolo_path'] else 'N/A'
  swin_name = config['swin_path'].name if config['swin_path'] else 'N/A'
  report += f"| {config['name']} | {yolo_name} | {config['yolo_imgsz']} | {swin_name} | {config['swin_imgsz']} | {config['backend']} |\n"

report += """
---

## 2. Results - Components

### 2.1 YOLO Detection

| Tier | Model | imgsz | Mean [ms] | P50 [ms] | P95 [ms] | P99 [ms] | FPS | VRAM [MB] |
|------|-------|-------|-----------|----------|----------|----------|-----|------------|
"""

for stats in component_results['yolo']:
  report += f"| {stats['tier']} | {stats['model']} | {stats['imgsz']} | {stats['mean_ms']:.2f} | {stats['p50_ms']:.2f} | {stats['p95_ms']:.2f} | {stats['p99_ms']:.2f} | {stats['fps_mean']:.1f} | {stats['vram_peak_mb']:.0f} |\n"

report += """
### 2.2 SWIN Classification

| Tier | Model | imgsz | Backend | Mean [ms] | P50 [ms] | P95 [ms] | P99 [ms] | FPS | VRAM [MB] |
|------|-------|-------|---------|-----------|----------|----------|----------|-----|------------|
"""

for stats in component_results['swin']:
  report += f"| {stats['tier']} | {stats['model']} | {stats['imgsz']} | {stats.get('backend', 'pytorch')} | {stats['mean_ms']:.2f} | {stats['p50_ms']:.2f} | {stats['p95_ms']:.2f} | {stats['p99_ms']:.2f} | {stats['fps_mean']:.1f} | {stats['vram_peak_mb']:.0f} |\n"

report += """
---

## 3. Results - End-to-End Pipeline

| Tier | Total Mean [ms] | Total P50 [ms] | Total P95 [ms] | YOLO [ms] | SWIN [ms] | FPS Mean | FPS P50 |
|------|-----------------|----------------|----------------|-----------|-----------|----------|----------|
"""

for result in pipeline_results:
  t = result['total']
  y = result['yolo']
  s = result['swin']
  report += f"| {result['tier']} | {t['mean_ms']:.2f} | {t['p50_ms']:.2f} | {t['p95_ms']:.2f} | {y['mean_ms']:.2f} | {s['mean_ms']:.2f} | {t['fps_mean']:.1f} | {t['fps_p50']:.1f} |\n"

report += """
---

## 4. Plots

### 4.1 Component FPS comparison
![Component FPS](plots/component_fps_comparison.png)

### 4.2 Component latency comparison (percentyle)
![Component Latency](plots/component_latency_percentiles.png)

### 4.3 Pipeline FPS
![Pipeline FPS](plots/pipeline_fps_comparison.png)

### 4.4 Latency vs number of objects
![Latency vs Objects](plots/latency_vs_objects.png)

### 4.5 Latency distribution
![Latency Distribution](plots/latency_distribution.png)

### 4.6 Breakdown latencji pipeline
![Pipeline Breakdown](plots/pipeline_latency_breakdown.png)

---

## 5. Wnioski

"""

# Automatyczne wnioski
if pipeline_results:
  fastest = max(pipeline_results, key=lambda x: x['total']['fps_mean'])
  slowest = min(pipeline_results, key=lambda x: x['total']['fps_mean'])

  report += f"""### Najszybszy tier: **{fastest['tier']}**
- Pipeline FPS: {fastest['total']['fps_mean']:.1f}
- Latencja P50: {fastest['total']['p50_ms']:.2f} ms

### Najwolniejszy tier: **{slowest['tier']}**
- Pipeline FPS: {slowest['total']['fps_mean']:.1f}
- Latencja P50: {slowest['total']['p50_ms']:.2f} ms

### Speedup
- {fastest['tier']} vs {slowest['tier']}: **{fastest['total']['fps_mean']/slowest['total']['fps_mean']:.2f}x** szybszy
"""

report += """
---

## 6. Tabele LaTeX

### Component table

```latex
\\begin{table}[htbp]
\\centering
\\caption{VSHORAD component performance comparison}
\\label{tab:component_latency}
\\begin{tabular}{llrrrrr}
\\toprule
Tier & Komponent & imgsz & Mean [ms] & P95 [ms] & FPS & VRAM [MB] \\\\
\\midrule
"""

for stats in component_results['yolo']:
  report += f"{stats['tier']} & YOLO & {stats['imgsz']} & {stats['mean_ms']:.2f} & {stats['p95_ms']:.2f} & {stats['fps_mean']:.1f} & {stats['vram_peak_mb']:.0f} \\\\\n"

for stats in component_results['swin']:
  report += f"{stats['tier']} & SWIN & {stats['imgsz']} & {stats['mean_ms']:.2f} & {stats['p95_ms']:.2f} & {stats['fps_mean']:.1f} & {stats['vram_peak_mb']:.0f} \\\\\n"

report += """\\bottomrule
\\end{tabular}
\\end{table}
```

### Pipeline table

```latex
\\begin{table}[htbp]
\\centering
\\caption{VSHORAD end-to-end pipeline performance}
\\label{tab:pipeline_latency}
\\begin{tabular}{lrrrrrr}
\\toprule
Tier & Mean [ms] & P50 [ms] & P95 [ms] & YOLO [ms] & SWIN [ms] & FPS \\\\
\\midrule
"""

for result in pipeline_results:
  t = result['total']
  y = result['yolo']
  s = result['swin']
  report += f"{result['tier']} & {t['mean_ms']:.2f} & {t['p50_ms']:.2f} & {t['p95_ms']:.2f} & {y['mean_ms']:.2f} & {s['mean_ms']:.2f} & {t['fps_mean']:.1f} \\\\\n"

report += """\\bottomrule
\\end{tabular}
\\end{table}
```

---

*Raport wygenerowany automatycznie przez notebook VSHORAD_6_3_Latency_Benchmark*
"""

# Save report
report_path = RUN_DIR / 'REPORT_6_3_LATENCY.md'
with open(report_path, 'w', encoding='utf-8') as f:
  f.write(report)

print(f" Report saved: {report_path}")

In [ ]:
# 6.3 Save results JSON

print(" Saving results JSON...")

all_results = {
  'metadata': {
    'timestamp': datetime.now().isoformat(),
    'gpu': gpu_name,
    'vram': gpu_vram,
    'cuda': str(torch.version.cuda) if torch.cuda.is_available() else None,
    'video': str(TEST_VIDEO) if TEST_VIDEO else None,
    'config': BENCHMARK_CONFIG,
  },
  'component_results': component_results,
  'pipeline_results': pipeline_results,
}

json_path = RUN_DIR / 'benchmark_results.json'
with open(json_path, 'w', encoding='utf-8') as f:
  json.dump(all_results, f, indent=2, ensure_ascii=False, default=str)

print(f" JSON saved: {json_path}")

In [ ]:
# 6.4 Final summary

print("\n" + "="*70)
print(" BENCHMARK COMPLETE - SUMMARY")
print("="*70)

print(f"\n Results saved to: {RUN_DIR}")
print(f"\n Contents:")

for item in sorted(RUN_DIR.rglob('*')):
  if item.is_file():
    rel_path = item.relative_to(RUN_DIR)
    size_kb = item.stat().st_size / 1024
    print(f"  {rel_path} ({size_kb:.1f} KB)")

print(f"\n Component results:")
print(f"  YOLO: {len(component_results['yolo'])} models tested")
print(f"  SWIN: {len(component_results['swin'])} models tested")

print(f"\n Pipeline results:")
for result in pipeline_results:
  print(f"  {result['tier']}: {result['total']['fps_mean']:.1f} FPS (P50: {result['total']['p50_ms']:.1f} ms)")

print("\n" + "="*70)
print(" Next steps:")
print("  1. Check report: REPORT_6_3_LATENCY.md")
print("  2. Copy plots to Overleaf")
print("  3. Use LaTeX tables from the report")
print("="*70)